# Running Nutrient Gradient Sweep Function for Infection Medium Tuning

```
see Obsidian Notes:
08.07.26, 08.08.26, 08.10.26, 08.11.26

continued from:
/home/gmvaz/2026_GEMs/stm_model_test/080726_081426.ipynb

tmux: gems
conda environment: gems
```

# Generate Constrained Models of the STM only condition and the Mixed STM condition

In [1]:
# Set the gurobi solver 

import os
from cobra.core.configuration import Configuration

os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

# Read in the gene weights CSV files

import pandas as pd

stm_threshold_10_90 = pd.read_csv("../imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("../imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("../imat_prep_test/stm_threshold_25_75.csv")

mixed_stm_threshold_10_90 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_25_75.csv")

# Make a python set of each gene weights CSV

stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

mixed_stm_set_10_90 = set(mixed_stm_threshold_10_90["genes"].astype(str).str.strip())
mixed_stm_set_15_85 = set(mixed_stm_threshold_15_85["genes"].astype(str).str.strip())
mixed_stm_set_25_75 = set(mixed_stm_threshold_25_75["genes"].astype(str).str.strip())

# QC: Check the count of genes

print("STM1090 gene count",len(stm_set_10_90), "|", "STM1585 gene count", len(stm_set_15_85), "|", "STM2575 gene count", len(stm_set_25_75))
print("Mixed STM1090 gene count",len(mixed_stm_set_10_90), "|", "Mixed STM1585 gene count", len(mixed_stm_set_15_85), "|", "Mixed STM2575 gene count", len(mixed_stm_set_25_75))

STM1090 gene count 4671 | STM1585 gene count 4671 | STM2575 gene count 4671
Mixed STM1090 gene count 4671 | Mixed STM1585 gene count 4671 | Mixed STM2575 gene count 4671


In [2]:
# Load STM_V1_0

from cobra.io import load_model, load_json_model

control_model = load_model("STM_v1_0")

# Save all genes in the control model in a set

model_genes = set([g.id for g in control_model.genes])

# QC: Check the count of genes
print("STM_V1_0 Control Model has", len(model_genes), "total genes")

# Subset the gene weights lists for each sample to keep only the genes also found in the control model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

mixed_stm1090_model = mixed_stm_threshold_10_90[mixed_stm_threshold_10_90["genes"].isin(model_genes)]
mixed_stm1585_model = mixed_stm_threshold_15_85[mixed_stm_threshold_15_85["genes"].isin(model_genes)]
mixed_stm2575_model = mixed_stm_threshold_25_75[mixed_stm_threshold_25_75["genes"].isin(model_genes)]

# Align the gene weights lists to the control model genes list and assign missing genes a weight of 0

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

subset_weights_mixedstm1090 = mixed_stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm1585 = mixed_stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm2575 = mixed_stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)


# QC: Confirm all samples have the same amount of gene weights

# 1. Make a list of the subset gene weights
subset_gene_weights = [
    subset_weights_stm1090,
    subset_weights_stm1585,
    subset_weights_stm2575,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
gene_counts = [len(weights) for weights in subset_gene_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(gene_counts)) == 1:
    print(f"All models have {gene_counts[0]} genes")
else:
    print("Gene counts differ between models")
    print(gene_counts)

# Convert gene weights to reaction weights with iMAT

from imatpy.parse_gpr import gene_to_rxn_weights

rxn_weights_stm1090 = gene_to_rxn_weights(control_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(control_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(control_model, subset_weights_stm2575)

rxn_weights_mixedstm1090 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1090)
rxn_weights_mixedstm1585 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1585)
rxn_weights_mixedstm2575 = gene_to_rxn_weights(control_model, subset_weights_mixedstm2575)

# QC: Confirm all samples have the same amount of reaction weights

# 1. Make a list of the reaction weights
reaction_weights = [
    rxn_weights_stm1090,
    rxn_weights_stm1585,
    rxn_weights_stm2575,
    rxn_weights_mixedstm1090,
    rxn_weights_mixedstm1585,
    rxn_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
reactions_counts = [len(weights) for weights in reaction_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(reactions_counts)) == 1:
    print(f"All models have {reactions_counts[0]} reactions")
else:
    print("Reaction counts differ between models")
    print(reactions_counts)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu
STM_V1_0 Control Model has 1271 total genes
All models have 1271 genes
All models have 2545 reactions


In [3]:
# Generate constrained models with iMAT by integrating the reaction weights

from imatpy.model_creation import generate_model

stm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

mixedstm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1090, method = "imat_restrictions")
mixedstm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1585, method = "imat_restrictions")
mixedstm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm2575, method = "imat_restrictions")

Read LP format model from file /tmp/tmp61wqnvlp.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpzo51h1pi.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpe3zmpght.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmphzi46rod.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpm6rm25qs.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmplbl_7o9k.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


In [4]:
stm1090_model

Name,STM_v1_0
Memory address,dfe3a502190
Number of metabolites,1802
Number of reactions,2545
Number of genes,1271
Number of groups,0
Objective expression,1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Compartments,"cytosol, periplasm, extracellular space"


# Make the `original_models` dicitonary of models

In [5]:
# Make a dictionary of the models 

original_models = {
    "stm_1090": stm1090_model,
    "stm_1585": stm1585_model,
    "stm_2575": stm2575_model,
    "mixed_stm1090": mixedstm1090_model,
    "mixed_stm1585": mixedstm1585_model,
    "mixed_stm2575": mixedstm2575_model
}

# Make a dictionary with copies of each original model

fba_models = {
    model_name: model.copy()
    for model_name, model in original_models.items()
}

Read LP format model from file /tmp/tmpfu7u8e6l.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmptlvkcy8_.lp
Reading time = 0.02 seconds
: 3201 rows, 6460 columns, 25176 nonzeros
Read LP format model from file /tmp/tmpe7hekncv.lp
Reading time = 0.02 seconds
: 3949 rows, 7128 columns, 28088 nonzeros
Read LP format model from file /tmp/tmp8vhmfv3e.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpdzjoalx6.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmpc7r4pyfm.lp
Reading time = 0.02 seconds
: 3901 rows, 7068 columns, 27884 nonzeros


In [ ]:
fba_models

{'stm_1090': <Model STM_v1_0 at 0xdfdb3558c10>,
 'stm_1585': <Model STM_v1_0 at 0xdfd1cfdb590>,
 'stm_2575': <Model STM_v1_0 at 0xdfd1f8c95d0>,
 'mixed_stm1090': <Model STM_v1_0 at 0xdfd18ee4950>,
 'mixed_stm1585': <Model STM_v1_0 at 0xdfd1466e790>,
 'mixed_stm2575': <Model STM_v1_0 at 0xdfd12eee290>}

# Define the Infection Medium Conditions

In [ ]:
# Make a copy of the default medium from any of the models (stm1090 used)

model_copy = fba_models.get("stm_1090")
default_medium = model_copy.medium

# Make 7 identical infection mediums with oxygen ranging from 0 - 3 at 0.5 increments

A_infection_medium = dict(default_medium)
A_infection_medium.update({
    "EX_o2_e": 0.0, # removing from default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

B_infection_medium = dict(default_medium)
B_infection_medium.update({
    "EX_o2_e": 0.5, # changing in default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

C_infection_medium = dict(default_medium)
C_infection_medium.update({
    "EX_o2_e": 1.0, # changing in the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

D_infection_medium = dict(default_medium)
D_infection_medium.update({
    "EX_o2_e": 1.5, # changing from the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

E_infection_medium = dict(default_medium)
E_infection_medium.update({
    "EX_o2_e": 2.0, # changing from the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

F_infection_medium = dict(default_medium)
F_infection_medium.update({
    "EX_o2_e": 2.5, # changing from the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

G_infection_medium = dict(default_medium)
G_infection_medium.update({
    "EX_o2_e": 3.0, # changing from the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

# Make `base_medias` - dicitonary of the infection medium's

In [ ]:
base_medias = {
	"A_infection_medium": A_infection_medium,
	"B_infection_medium": B_infection_medium,
	"C_infection_medium": C_infection_medium,
	"D_infection_medium": D_infection_medium,
	"E_infection_medium": E_infection_medium,
	"F_infection_medium": F_infection_medium,
	"G_infection_medium": G_infection_medium
}

# Make the `nutrient_rxn_list` -  list of reaction_ID's who's value will be changed

In [ ]:
nutrient_rxn_list = list(A_infection_medium.keys())
nutrient_rxn_list.remove('EX_o2_e')
nutrient_rxn_list

['EX_ca2_e',
 'EX_cbl1_e',
 'EX_cl_e',
 'EX_co2_e',
 'EX_cobalt2_e',
 'EX_cu2_e',
 'EX_mg2_e',
 'EX_fe2_e',
 'EX_fe3_e',
 'EX_mn2_e',
 'EX_mobd_e',
 'EX_na1_e',
 'EX_nh4_e',
 'EX_pi_e',
 'EX_zn2_e',
 'EX_so4_e',
 'EX_glc__D_e',
 'EX_tungs_e',
 'EX_h2o_e',
 'EX_h_e',
 'EX_k_e',
 'EX_tet_e',
 'EX_no3_e',
 'EX_etha_e',
 'EX_12ppd__R_e',
 'EX_12ppd__S_e',
 'EX_lac__L_e',
 'EX_for_e',
 'EX_succ_e',
 'EX_asp__L_e',
 'EX_mal__L_e',
 'EX_galct__D_e',
 'EX_glcr_e',
 'EX_pyr_e',
 'EX_arab__L_e',
 'EX_but_e',
 'EX_ac_e',
 'EX_ppa_e']

# Make the `nutrient_step_count_list`

In [ ]:
import numpy as np

nutrient_step_count = np.logspace(0.5, 7, num = 15, base = 2.0)

nutrient_step_count_array = np.insert(nutrient_step_count, 0, 0) # add values 0.5 to list of nutrient step count

nutrient_step_count_list = nutrient_step_count_array.tolist()

nutrient_step_count_list

[0.0,
 1.4142135623730951,
 1.9510972841051464,
 2.6918003852647123,
 3.7136996566701033,
 5.12354675905709,
 7.06861992598028,
 9.752109233635803,
 13.454342644059432,
 18.562070178562607,
 25.608865362590898,
 35.330864437562006,
 48.74366607935861,
 67.24842487380603,
 92.77822149538744,
 128.0]

# Define `run_medium_sweep()`

In [ ]:

def run_medium_sweep(*, model_name, medium_name, nutrient_rxn, nutrient_step_count): 
    
    results = [ ]
    
    for nutrient_value in nutrient_step_count:
        
        model = fba_models[model_name].copy()
        medium = base_medias[medium_name].copy()
        medium[nutrient_rxn] = float(nutrient_value)
        
        try:
            with model:
                model.medium = medium
                solution = model.optimize()

                if solution.status == "optimal": 
                    objective_value = solution.objective_value
                else:
                    objective_value = np.nan # NaN - Not a Number

                results.append({
                    "model": model_name,
                    "medium_condition": medium_name,
                    "nutrient_rxn": nutrient_rxn,
                    "nutrient_rxn_value": float(nutrient_value),
                    "status": solution.status,
                    "objective_value": objective_value
                })
                
        except Exception as error:
            results.append({
                "model": model_name,
                "medium_condition": medium_name,
                "nutrient_rxn": nutrient_rxn,
                "nutrient":float(nutrient_value),
                "status": "failed",
                "objective_value": np.nan,
                "error": str(error)
            })

    results_df = pd.DataFrame(results)

    return results_df

In [12]:
stm1090_ca2_E_medium_results = run_medium_sweep(model_name = "stm_1090", medium_name = "E_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
stm1090_ca2_E_medium_results.to_csv("nutrient_sweep_081226/stm1090_ca2_E_medium_results.csv")

Read LP format model from file /tmp/tmpky9msi1o.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpp4imybz3.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros


Read LP format model from file /tmp/tmp98ro5yte.lp
Reading time = 0.04 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpcijotfzn.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmplw6tdguy.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpbduv21h0.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpu77qxel4.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp53zg3nlp.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpalryuti1.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmplfk50daa.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /

# Define `run_medium_sweep_fast()` with `model.slim_optimize()` instead of `model.optimize`

In [ ]:
def run_medium_sweep_fast(*, model_name, medium_name, nutrient_rxn, nutrient_step_count): 
    
    results = [ ]
    
    for nutrient_value in nutrient_step_count:
        
        model = fba_models[model_name].copy()
        medium = base_medias[medium_name].copy()
        medium[nutrient_rxn] = float(nutrient_value)
        
        try:
            with model:
                model.medium = medium
                objective_value = model.slim_optimize()

                if isinstance(objective_value, float) == True:
                        status = "optimal"
                else:
                    status = "infeasible"

                results.append({
                    "model": model_name,
                    "medium_condition": medium_name,
                    "nutrient_rxn": nutrient_rxn,
                    "nutrient_rxn_value": float(nutrient_value),
                    "status": status,
                    "objective_value": objective_value
                })
                
        except Exception as error:
            #raise
            results.append({
                "model": model_name,
                "medium_condition": medium_name,
                "nutrient_rxn": nutrient_rxn,
                "nutrient":float(nutrient_value),
                "status": "failed",
                "objective_value": np.nan,
                "error": str(error)
            })

    results_df = pd.DataFrame(results)

    return results_df

In [15]:
# test the run_medium_sweep_fast function

# stm1090_cbl1_A_medium_results = run_medium_sweep_fast(model_name = "stm_1090", medium_name = "A_infection_medium", nutrient_rxn = "EX_cbl1_e", nutrient_step_count = nutrient_step_count_list)
# stm1090_cbl1_A_medium_results.to_csv("nutrient_sweep_081226/stm1090_cbl1_A_medium_results.csv")

## 08.18.26

Meeting with Titus 

Round objective-value to three digits. 

## 09.08.26 - Get all the exchange reactions from control_model

In [ ]:
dir(control_model)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_annotation',
 '_compartments',
 '_contexts',
 '_id',
 '_populate_solver',
 '_repr_html_',
 '_sbml',
 '_set_id_with_model',
 '_solver',
 '_tolerance',
 'add_boundary',
 'add_cons_vars',
 'add_groups',
 'add_metabolites',
 'add_reactions',
 'annotation',
 'boundary',
 'compartments',
 'constraints',
 'copy',
 'demands',
 'exchanges',
 'genes',
 'get_associated_groups',
 'groups',
 'id',
 'medium',
 'merge',
 'metabolites',
 'name',
 'notes',
 'objective',
 'objective_direction',
 'optimize',
 'problem',
 'reactions',
 'remove_cons_vars',
 'remove_groups',
 'remo

In [22]:
control_model.reactions

[<Reaction 3HAD180 at 0xdfdb2a45710>,
 <Reaction 3HAD181 at 0xdfdb2a456d0>,
 <Reaction 3HAD40 at 0xdfdb2a463d0>,
 <Reaction 3HAD60 at 0xdfdb2a47290>,
 <Reaction 3HAD80 at 0xdfdb2a47f90>,
 <Reaction 3KGK at 0xdfdb2a466d0>,
 <Reaction 3NTD2pp at 0xdfdb2a47f10>,
 <Reaction 3NTD4pp at 0xdfdb2a48dd0>,
 <Reaction 3NTD7pp at 0xdfdb2a49dd0>,
 <Reaction 3NTD9pp at 0xdfdb2a4b150>,
 <Reaction 3OAR100 at 0xdfdb2a4bf90>,
 <Reaction 3OAR120 at 0xdfdb2a51110>,
 <Reaction 3OAR121 at 0xdfdb2a52810>,
 <Reaction 3OAR140 at 0xdfdb2a53e90>,
 <Reaction 3OAR141 at 0xdfdb2a5cd50>,
 <Reaction 3OAR160 at 0xdfdb2a5dbd0>,
 <Reaction 3OAR161 at 0xdfdb2a49350>,
 <Reaction 23CAMPtex at 0xdfdb2a5f790>,
 <Reaction 23CCMPtex at 0xdfdb2a49050>,
 <Reaction 23CGMPtex at 0xdfdb2a613d0>,
 <Reaction 12DGR120tipp at 0xdfdb2a51150>,
 <Reaction 12DGR140tipp at 0xdfdb2a62790>,
 <Reaction 12DGR141tipp at 0xdfdb2a68510>,
 <Reaction 12DGR160tipp at 0xdfdb2a695d0>,
 <Reaction 23CUMPtex at 0xdfdb2a69a90>,
 <Reaction 3OAR180 at 0xdfdb

In [23]:
type(control_model.reactions)

cobra.core.dictlist.DictList

`control_model` is a a `cobra.core.model.Model` object. The model has has a reaction attribute who's type is `cobra.core.dictlist.DictList`. When I print `control_model.reactions` shows the reaction ID and it's location. What is a `dictlist`, what can I do with that type of object, and how do I copy only exchange reactions from it? 

In [27]:
import inspect
inspect.getmembers(control_model.reactions)

[('12DGR120tipp', <Reaction 12DGR120tipp at 0xdfdb2a51150>),
 ('12DGR140tipp', <Reaction 12DGR140tipp at 0xdfdb2a62790>),
 ('12DGR141tipp', <Reaction 12DGR141tipp at 0xdfdb2a68510>),
 ('12DGR160tipp', <Reaction 12DGR160tipp at 0xdfdb2a695d0>),
 ('12DGR161tipp', <Reaction 12DGR161tipp at 0xdfdb2a6b810>),
 ('12DGR180tipp', <Reaction 12DGR180tipp at 0xdfdb2a78b10>),
 ('12DGR181tipp', <Reaction 12DGR181tipp at 0xdfdb2a78990>),
 ('12PPDRDH', <Reaction 12PPDRDH at 0xdfdb15eca90>),
 ('12PPDRtex', <Reaction 12PPDRtex at 0xdfdb2a799d0>),
 ('12PPDRtpp', <Reaction 12PPDRtpp at 0xdfdb2a473d0>),
 ('12PPDStex', <Reaction 12PPDStex at 0xdfdb2a7a690>),
 ('12PPDStpp', <Reaction 12PPDStpp at 0xdfdb2a7a4d0>),
 ('12dgr2_ST', <Reaction 12dgr2_ST at 0xdfdb139a210>),
 ('14GLUCANabcpp', <Reaction 14GLUCANabcpp at 0xdfdb2a5f890>),
 ('14GLUCANtexi', <Reaction 14GLUCANtexi at 0xdfdb2a93bd0>),
 ('23CAMPtex', <Reaction 23CAMPtex at 0xdfdb2a5f790>),
 ('23CCMPtex', <Reaction 23CCMPtex at 0xdfdb2a49050>),
 ('23CGMPte

What's the difference between `inspect.control_model.reactions` and `control_model.reactions`?

In [54]:
control_model.exchanges['EX_alaala_e']

TypeError: list indices must be integers or slices, not str

In [32]:
help(control_model.exchanges)

Help on DictList in module cobra.core.dictlist object:

class DictList(builtins.list)
 |  DictList(*args)
 |  
 |  Define a combined dict and list.
 |  
 |  This object behaves like a list, but has the O(1) speed
 |  benefits of a dict when looking up elements by their id.
 |  
 |  Method resolution order:
 |      DictList
 |      builtins.list
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __add__(self, other: Iterable[cobra.core.object.Object]) -> 'DictList'
 |      Add item while returning a new DictList.
 |      
 |      x.__add__(y) <==> x + y
 |      
 |      Parameters
 |      ----------
 |      other : iterable
 |          other must contain only unique id's which do not intersect
 |          with self
 |  
 |  __contains__(self, entity: Union[str, cobra.core.object.Object]) -> bool
 |      Ask if the DictList contain an entity.
 |      
 |      DictList.__contains__(entity) <==> entity in DictList
 |      
 |      Parameters
 |      ----------
 |      entity: 

`control.model.exchanges` is a dictlist of all exchange reactions. It has the reaction ID and position (?) but I want a list of only reaction IDs. 

In [38]:
control_model.exchanges[0]

Reaction identifier,EX_alaala_e
Name,D-Alanyl-D-alanine exchange
Memory address,0xdfdb23c5710
Stoichiometry,alaala_e --> D-Alanyl-D-alanine -->
GPR,
Lower bound,0.0
Upper bound,1000.0


In [40]:
type(control_model.exchanges[0])

cobra.core.reaction.Reaction

In [45]:
dir(control_model.exchanges[0])

['__add__',
 '__class__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__iadd__',
 '__imul__',
 '__init__',
 '__init_subclass__',
 '__isub__',
 '__le__',
 '__lt__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__radd__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 '__weakref__',
 '_annotation',
 '_associate_gene',
 '_check_bounds',
 '_dissociate_gene',
 '_genes',
 '_gpr',
 '_id',
 '_lower_bound',
 '_metabolites',
 '_model',
 '_repr_html_',
 '_set_id_with_model',
 '_update_awareness',
 '_upper_bound',
 'add_metabolites',
 'annotation',
 'boundary',
 'bounds',
 'build_reaction_from_string',
 'build_reaction_string',
 'check_mass_balance',
 'compartments',
 'copy',
 'delete',
 'flux',
 'flux_expression',
 'forward_variable',
 'functional',
 'gene_name

Above are attributes and methods of a reaction in the the exchange reaction list of the model. There's an `id` attribute or method, I can't tell just by looking at it. 

In [55]:
# Why is it that I get the same thing for `control_model.exchanges[0].id` and `control_model.exchanges[0]._id`?
control_model.exchanges[0].id

'EX_alaala_e'

In [57]:
nutrient_rxn_list = []
for rxn in control_model.exchanges:
   nutrient_rxn_list.append(rxn.id)

In [58]:
nutrient_rxn_list

['EX_alaala_e',
 'EX_all__D_e',
 'EX_alltn_e',
 'EX_amp_e',
 'EX_anhgm_e',
 'EX_arab__L_e',
 'EX_arbtn_e',
 'EX_arbtn_fe3_e',
 'EX_chitob_e',
 'EX_pydxn_e',
 'EX_pydx_e',
 'EX_arg__L_e',
 'EX_ascb__L_e',
 'EX_asn__L_e',
 'EX_12ppd__R_e',
 'EX_aso3_e',
 'EX_asp__L_e',
 'EX_12ppd__S_e',
 'EX_but_e',
 'EX_14glucan_e',
 'EX_15dap_e',
 'EX_butso3_e',
 'EX_ca2_e',
 'EX_cbi_e',
 'EX_cbl1_e',
 'EX_23camp_e',
 'EX_23ccmp_e',
 'EX_23cgmp_e',
 'EX_cd2_e',
 'EX_23cump_e',
 'EX_23dappa_e',
 'EX_26dap__M_e',
 'EX_cgly_e',
 'EX_chol_e',
 'EX_cit_e',
 'EX_2ddglcn_e',
 'EX_34dhpac_e',
 'EX_3amp_e',
 'EX_3cmp_e',
 'EX_3gmp_e',
 'EX_icit_e',
 'EX_acon_C_e',
 'EX_cl_e',
 'EX_cmp_e',
 'EX_co2_e',
 'EX_cobalt2_e',
 'EX_colipa_e',
 'EX_3hcinnm_e',
 'EX_3hpppn_e',
 'EX_cpgn_e',
 'EX_cpgn_un_e',
 'EX_crn_e',
 'EX_3ump_e',
 'EX_csn_e',
 'EX_cu2_e',
 'EX_cu_e',
 'EX_4abut_e',
 'EX_4hoxpacd_e',
 'EX_5dglcn_e',
 'EX_cyan_e',
 'EX_LalaDgluMdapDala_e',
 'EX_LalaDgluMdap_e',
 'EX_ac_e',
 'EX_cynt_e',
 'EX_cys__D_e',


In [ ]:
# Make the `nutrient_rxn_list` -  list of reaction_ID's who's value will be changed

nutrient_rxn_list = list(A_infection_medium.keys())
nutrient_rxn_list.remove('EX_o2_e')
nutrient_rxn_list
